> **Executed on Databricks** (Free Edition, serverless compute, Unity Catalog).  
> Job run `1005795510073716` · task `silver_to_gold` · notebook `05_silver_to_gold` · started 2026-11-17 20:35 UTC · result **SUCCESS**  
> Parameters: `catalog={{job.parameters.catalog}}`

Saved with `scripts/export_databricks_run.py`: the cells and outputs below are exactly what the run produced.

# 05 · Silver → Gold
Builds the star schema (`dim_date`, `dim_tag`, `dim_user`, `fact_question`, `fact_answer`,
`bridge_question_tag`) and the dashboard aggregates (`agg_tag_month`, `agg_tag_pairs`, `agg_user_cohort`).

Deleted questions are excluded. Each table is refreshed with `MERGE INTO` from the **current** Silver
layer, so this notebook does not take a batch id: run it at the end of every incremental load and Gold
follows Silver. A second run on the same Silver data changes nothing. `OPTIMIZE` then compacts the
fact and aggregate tables (liquid clustering on Databricks).

| Widget | Meaning |
|---|---|
| `catalog` | Unity Catalog. Free Edition uses `workspace`. |

In [1]:
dbutils.widgets.text("catalog", "workspace")

In [2]:
%run ./_bootstrap

repo root: /Workspace/Users/<your-email>/stackoverflow-lakehouse


In [3]:
from so_lakehouse.gold import run_silver_to_gold

cfg = build_config()
audit = run_silver_to_gold(spark, cfg, fail_on_error=True)

[SUCCESS] Silver-to-Gold | dim_date                           | read=6606 ins=41 upd=0 del=0 quar=0 | 14.3s | calendar 2008-10-17 to 2026-11-17
[SUCCESS] Silver-to-Gold | dim_tag                            | read=2822 ins=11 upd=0 del=0 quar=0 | 12.9s | 18 tracked tags, other tags categorised as Other
[SUCCESS] Silver-to-Gold | dim_user                           | read=8391 ins=59 upd=17 del=0 quar=0 | 12.3s | reputation band and cohort month derived
[SUCCESS] Silver-to-Gold | fact_question                      | read=7652 ins=32 upd=1 del=0 quar=0 | 17.2s | aggregated 7651 current questions; excluded 1 deleted
[SUCCESS] Silver-to-Gold | fact_answer                        | read=9658 ins=49 upd=0 del=0 quar=0 | 15.6s | aggregated 9658 current answers
[SUCCESS] Silver-to-Gold | bridge_question_tag                | read=24295 ins=113 upd=0 del=0 quar=0 | 14.4s | aggregated 24295 question-tag pairs
[SUCCESS] Silver-to-Gold | agg_tag_month                      | read=8271 ins=20 upd=40 del

In [4]:
display(spark.table(cfg.table("ops.pipeline_execution_logs"))
        .where(f"run_id = '{audit.run_id}'").orderBy("start_time"))

log_id,run_id,pipeline_name,layer,run_mode,load_type,entity,batch_id,source_parameter,target_table,start_time,end_time,duration_seconds,status,rows_read,rows_inserted,rows_updated,rows_deleted,rows_quarantined,message,load_timestamp
1cf24964-e2ee-4bc3-a2ae-eb4e72cbff23,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,dim_date,,current Silver layer,workspace.so_gold.dim_date,2026-11-17T20:35:16.360Z,2026-11-17T20:35:30.662Z,14.301579,SUCCESS,6606,41,0,0,0,calendar 2008-10-17 to 2026-11-17,2026-11-17T20:35:30.662Z
1e9190dd-569e-41a5-8c33-91d787dc7ba4,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,dim_tag,,current Silver layer,workspace.so_gold.dim_tag,2026-11-17T20:35:32.905Z,2026-11-17T20:35:45.782Z,12.877647,SUCCESS,2822,11,0,0,0,"18 tracked tags, other tags categorised as Other",2026-11-17T20:35:45.782Z
b42ca521-9ca4-411f-acd3-3f48ca85bea2,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,dim_user,,current Silver layer,workspace.so_gold.dim_user,2026-11-17T20:35:47.782Z,2026-11-17T20:36:00.050Z,12.268097,SUCCESS,8391,59,17,0,0,reputation band and cohort month derived,2026-11-17T20:36:00.050Z
089ffda8-a3e3-4041-ac29-aa25fd21e5c7,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,fact_question,,current Silver layer,workspace.so_gold.fact_question,2026-11-17T20:36:01.760Z,2026-11-17T20:36:19.007Z,17.246333,SUCCESS,7652,32,1,0,0,aggregated 7651 current questions; excluded 1 deleted,2026-11-17T20:36:19.007Z
233dd7dd-4025-4d4c-9f40-f798bf5df08c,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,fact_answer,,current Silver layer,workspace.so_gold.fact_answer,2026-11-17T20:36:21.093Z,2026-11-17T20:36:36.648Z,15.554159,SUCCESS,9658,49,0,0,0,aggregated 9658 current answers,2026-11-17T20:36:36.648Z
ca8b6509-e8f3-4144-aab2-7ec15e7d75de,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,bridge_question_tag,,current Silver layer,workspace.so_gold.bridge_question_tag,2026-11-17T20:36:38.757Z,2026-11-17T20:36:53.132Z,14.37501,SUCCESS,24295,113,0,0,0,aggregated 24295 question-tag pairs,2026-11-17T20:36:53.132Z
0012c095-4d67-4346-ae95-a60cb08cdb37,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,agg_tag_month,,current Silver layer,workspace.so_gold.agg_tag_month,2026-11-17T20:36:55.487Z,2026-11-17T20:37:23.004Z,27.517664,SUCCESS,8271,20,40,0,0,aggregated 8271 question-tag rows into 377 tag-months,2026-11-17T20:37:23.004Z
b0bec166-27c9-44fc-ac44-78f674aac921,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,agg_tag_pairs,,current Silver layer,workspace.so_gold.agg_tag_pairs,2026-11-17T20:37:25.716Z,2026-11-17T20:37:42.237Z,16.521142,SUCCESS,8271,0,4,0,0,aggregated 62 tracked tag pairs,2026-11-17T20:37:42.237Z
36cf6393-2d17-477a-8eba-067f8a01f5eb,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,agg_user_cohort,,current Silver layer,workspace.so_gold.agg_user_cohort,2026-11-17T20:37:44.006Z,2026-11-17T20:38:04.097Z,20.091459,SUCCESS,8389,288,1392,0,0,aggregated 8389 users into 17419 cohort cells,2026-11-17T20:38:04.097Z
6ab81f42-c2c2-4d9e-807a-e6d841951ee3,silver_to_gold-20261117T203516Z-4808ba,silver_to_gold,Silver-to-Gold,refresh,,optimize,,current Silver layer,-,2026-11-17T20:38:06.076Z,2026-11-17T20:38:43.708Z,37.632189,SUCCESS,0,0,0,0,0,OPTIMIZE gold.fact_question CLUSTER BY (created_date_key); gold.fact_answer CLUSTER BY (created_date_key); gold.bridge_question_tag CLUSTER BY (tag); gold.agg_tag_month CLUSTER BY (month_start),2026-11-17T20:38:43.708Z


In [5]:
for logical in ("gold.fact_question", "gold.fact_answer", "gold.dim_tag", "gold.agg_tag_month"):
    print(logical, spark.table(cfg.table(logical)).count())

gold.fact_question 7651
gold.fact_answer 9658
gold.dim_tag 2822
gold.agg_tag_month 377
